# NumPy：ML 需要的陣列操作

已學過 vibe coding，本篇聚焦在能讀懂、檢查 AI 產生的資料運算。
不需背 API；要能說出資料的 shape、運算方向與輸出意義。
套件安裝請見 [使用說明](./README.md)。

In [ ]:
import numpy as np

## 1. 先看 shape：樣本 × 特徵

ML 常用 `X.shape == (樣本數, 特徵數)`，`y` 是每筆樣本對應的答案。
`dtype` 用來確認數值型態；本篇不逐一介紹整數位元範圍。

In [ ]:
X = np.array([[18, 90], [25, 75], [40, 60], [32, 85]], dtype=float)
y = np.array([0, 1, 1, 0])
print("X:", X.shape, "y:", y.shape, "dtype:", X.dtype)

## 2. 取資料：列是樣本，欄是特徵

`X[列, 欄]`；切片不含結束位置。保留單一特徵時，注意 `(n,)` 與 `(n, 1)` 的差別。

In [ ]:
print("前兩筆樣本：\n", X[:2])
print("第一個特徵，一維：", X[:, 0].shape)
print("第一個特徵，二維：", X[:, :1].shape)
print("第一個特徵 >= 30 的樣本：\n", X[X[:, 0] >= 30])

## 3. 整批運算與 axis

陣列可以整批計算，不必逐筆寫迴圈。
對二維資料，`axis=0` 跨樣本、得到每個特徵的統計值；`axis=1` 跨特徵、得到每筆樣本的統計值。
混合不同單位的特徵時，跨特徵平均未必有意義。

In [ ]:
print("每個特徵的平均：", X.mean(axis=0))
print("每個特徵的標準差：", X.std(axis=0))
scores = np.array([[80, 90], [60, 70]])
print("每位學生的平均成績：", scores.mean(axis=1))
print("成績加 5 分：\n", scores + 5)

### 組合條件並保持標籤對應

判斷式會產生布林陣列。用 `&` 組合兩個條件時，每個條件都要加上括號；
用相同條件篩選樣本和標籤，就能保持兩者的順序與對應關係。
以下整理自原教材的 NumPy 判斷式取值與 pandas 多條件篩選範例。

In [ ]:
mask_demo = (X[:, 0] >= 20) & (X[:, 1] >= 80)
print('符合條件：', mask_demo)
print('篩選後的樣本：', X[mask_demo])
print('對應的標籤：', y[mask_demo])

### 小練習 3：篩選樣本並保持標籤對齊

使用上面的 `X` 和 `y`，找出第一個特徵至少為 25、且第二個特徵至少為 75 的樣本。
用同一個條件取出對應標籤，印出篩選後的樣本、標籤、各自的 shape，
以及篩選後兩個特徵各自的平均值。

<details>
<summary>參考答案</summary>

```python
mask = (X[:, 0] >= 25) & (X[:, 1] >= 75)
selected_X = X[mask]
selected_y = y[mask]
print(selected_X, selected_X.shape)
print(selected_y, selected_y.shape)
print('每個特徵的平均：', selected_X.mean(axis=0))
```

選出 `[25, 75]` 和 `[32, 85]`，對應標籤為 `[1, 0]`；shape 為 `(2, 2)` 和 `(2,)`，特徵平均為 `[28.5, 80]`。

</details>

## 4. Broadcasting：每個特徵用自己的統計值

`(4, 2)` 減去 `(2,)`，會把兩個平均值套用到每一筆樣本。
判斷 shape 是否相容：從最右邊比起，大小要相等，或其中一邊為 1；缺少的維度視為 1。

以下示範每個特徵減平均、除標準差。實際 ML 要先切分資料，統計值只從訓練集估計，
驗證／測試集沿用它們，避免資料洩漏。

In [ ]:
mean = X.mean(axis=0)
std = X.std(axis=0)
X_scaled = (X - mean) / np.where(std == 0, 1, std)
print("輸出 shape：", X_scaled.shape)
print("標準化後每個特徵的平均：", X_scaled.mean(axis=0))

## 5. 矩陣乘法：讀懂線性模型

`*` 是對應位置相乘；`@` 是矩陣乘法。
線性模型 `X @ w + b` 將每筆樣本的特徵加權後，得到一個預測值。
`(4, 2) @ (2,) → (4,)`，中間的特徵數必須一致。

In [ ]:
w = np.array([0.2, -0.1])
b = 1.0
pred = X @ w + b
print("逐元素相乘的 shape：", (X * w).shape)
print("預測：", pred, "shape：", pred.shape)

### 用布林陣列檢查分類結果

比較預測與標籤可得到每筆是否正確的布林值；布林值的平均數是正確比例。
把布林條件交給 `np.array(..., dtype=int)` 可轉成 0／1，
再用條件篩選檢查個別錯誤。這裡沿用原教材的判斷式取值與平均數運算。

In [ ]:
actual_demo = np.array([1, 0, 1])
pred_demo = np.array([1, 1, 0])
print('是否正確：', pred_demo == actual_demo)
print('正確比例：', np.mean(pred_demo == actual_demo))
print('把布林值轉成 0/1：', np.array(pred_demo > 0, dtype=int))
print('漏判的標籤：', actual_demo[(actual_demo == 1) & (pred_demo == 0)])

### 小練習 4：從模型分數檢查預測錯誤

使用上面的 `X`、`y`、`w` 和 `b` 計算線性分數 `X @ w + b`。
將分數大於或等於 0 的樣本預測為 1，其餘預測為 0，印出預測結果、準確率，
以及真實標籤為 1 卻被預測為 0 的樣本。這裡的分數不用解讀為機率。

<details>
<summary>參考答案</summary>

```python
scores_ex = X @ w + b
predicted_ex = np.array(scores_ex >= 0, dtype=int)
accuracy_ex = np.mean(predicted_ex == y)
missed = X[(y == 1) & (predicted_ex == 0)]
print('預測：', predicted_ex)
print('準確率：', accuracy_ex)
print('漏判樣本：', missed)
```

預測為 `[0, 0, 1, 0]`，準確率為 0.75；漏判樣本為 `[[25, 75]]`。

</details>

## 6. reshape 與可重現的範例

單一特徵常需要用 `reshape(-1, 1)` 轉成二維；`-1` 表示自動推算大小，元素總數不變。
`reshape` 不會替你決定樣本和特徵的意義。固定亂數種子便於比較結果。

In [ ]:
age = X[:, 0]
print("轉換前後：", age.shape, age.reshape(-1, 1).shape)
rng = np.random.default_rng(42)
print("模擬資料：\n", rng.normal(size=(3, 2)))

## 用 AI 協作時，先驗收這三件事

- 輸入和輸出的 shape 是否符合任務？
- 統計值是按特徵算，還是按樣本算？
- 使用的是逐元素運算，還是矩陣乘法？

接下來保留兩個 ML 練習。可請 AI 協助實作，但需解釋公式與輸出。
softmax 的輸出應非負且總和約為 1；Linear Layer 的輸出維度應是 `d_out`，經 ReLU 後不應有負值。

## 練習

### 練習 1：softmax 函數

實作 Deep Learning 中常用的機率輸出函數 softmax。

若當前任務為 $c$ 分類問題，輸入資料 $D$ 經過神經網路計算後得到的 logits 為 $x \in R^c$，則分類 $i$ 的 logits $x_i$ 經過 softmax 得到的結果 $p_i$ 為：

$$
p_i = \text{softmax}(x_i) = \frac{e^{x_i}}{\sum_{j = 1}^c e^{x_j}}
$$

### 練習 2：Linear Layer + ReLU Activation

實作 Deep Learning 中線性神經網路與常用的激發函數（Activation Function）ReLU。

令當前神經網路權重（Weight）為 $W \in R^{d_{out} \times d_{in}}$，偏差值（Bias）為 $b \in R^{d_{out}}$。
若神經網路輸入值為 $x \in R^{d_{in}}$，則輸出值 $y \in R^{d_{out}}$ 為：

$$
y = \text{ReLU}(Wx + b) = \max(0, Wx + b)
$$

In [ ]:
# 練習 1 解答

# 設定總共類別
c = 10                    
# 模擬輸出 logits
x = np.random.rand(c)     
# 計算指數
x_exp = np.exp(x)         
# 取得 softmax 分母
x_exp_sum = np.sum(x_exp) 
# 定義機率值維度，儲存 softmax 結果
p = np.zeros_like(x)      

# 對每個維度計算 softmax
for i in range(c):        
    p[i] = x_exp[i] / x_exp_sum

# 輸出 logits
print(x)                  
# 輸出經過 softmax 後的機率值
print(p)                  

In [ ]:
# 練習 2 解答

# 設定輸入維度
d_in = 10                                
# 設定輸出維度
d_out = 30                               

# 模擬神經網路輸入
x = np.ones((d_in, 1))                   
# 模擬神經網路權重
W = np.random.rand(d_out, d_in) * 10 - 5 
# 模擬神經網路偏差值
b = np.random.rand(d_out, 1) * 10 - 5    

h = W.dot(x) + b
# 計算 ReLU 得到神經網路輸出
y = h * (h > 0)                          

# 輸出神經網路輸入
print(x)                                 
# 輸出神經網路輸出
print(y)                                 